In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/raw")

customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")

In [3]:
datasets = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "products": products,
    "payments": payments,
    "reviews": reviews
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
items: (112650, 7)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)


In [4]:
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(df.isnull().sum())


--- CUSTOMERS ---
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

--- ORDERS ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

--- ITEMS ---
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

--- PRODUCTS ---
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm         

In [5]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [6]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_day"] = orders["order_purchase_timestamp"].dt.day

In [7]:
orders[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_time_days",
    "order_estimated_delivery_date",
    "delivery_delay_days",
    "purchase_year",
    "purchase_month",
    "purchase_day"
]].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days,order_estimated_delivery_date,delivery_delay_days,purchase_year,purchase_month,purchase_day
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0,2017-10-18,-8.0,2017,10,2
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0,2018-08-13,-6.0,2018,7,24
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0,2018-09-04,-18.0,2018,8,8
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0,2017-12-15,-13.0,2017,11,18
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0,2018-02-26,-10.0,2018,2,13


In [8]:
print("Orders rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())

print()

print("Items rows:", len(items))
print("Unique order IDs in items:", items["order_id"].nunique())

Orders rows: 99441
Unique order IDs: 99441

Items rows: 112650
Unique order IDs in items: 98666


In [9]:
print("Customer rows:", len(customers))
print("Unique customer IDs:", customers["customer_id"].nunique())

Customer rows: 99441
Unique customer IDs: 99441


In [10]:
orders_customers = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

In [11]:
print("Before merge:", orders.shape)
print("After merge:", orders_customers.shape)

orders_customers.head()

Before merge: (99441, 13)
After merge: (99441, 17)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,purchase_year,purchase_month,purchase_day,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,2017,10,2,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,2018,7,24,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,2018,8,8,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,2017,11,18,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,2018,2,13,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [12]:
order_items = orders_customers.merge(
    items,
    on="order_id",
    how="left"
)

print("Before:", orders_customers.shape)
print("After:", order_items.shape)

Before: (99441, 17)
After: (113425, 23)


In [13]:
print("Product rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())

Product rows: 32951
Unique product IDs: 32951


In [14]:
order_items_products = order_items.merge(
    products,
    on="product_id",
    how="left"
)

print("Before:", order_items.shape)
print("After:", order_items_products.shape)

Before: (113425, 23)
After: (113425, 31)


In [15]:
print("Payment rows:", len(payments))
print("Unique order IDs:", payments["order_id"].nunique())

Payment rows: 103886
Unique order IDs: 99440


In [16]:
payment_summary = payments.groupby("order_id").agg(
    total_payment=("payment_value", "sum"),
    payment_count=("payment_sequential", "count"),
    max_installments=("payment_installments", "max")
).reset_index()

payment_summary.head()

,order_id,total_payment,payment_count,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


In [17]:
print("Rows:", len(payment_summary))
print("Unique orders:", payment_summary["order_id"].nunique())

Rows: 99440
Unique orders: 99440


In [18]:
final_df = order_items_products.merge(
    payment_summary,
    on="order_id",
    how="left"
)

print("Before:", order_items_products.shape)
print("After:", final_df.shape)

Before: (113425, 31)
After: (113425, 34)


In [19]:
print("Review rows:", len(reviews))
print("Unique order IDs:", reviews["order_id"].nunique())

Review rows: 99224
Unique order IDs: 98673


In [20]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [21]:
review_summary = reviews.groupby("order_id").agg(
    avg_review_score=("review_score", "mean"),
    review_count=("review_score", "count")
).reset_index()

review_summary.head()

,order_id,avg_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [22]:
print("Rows:", len(review_summary))
print("Unique orders:", review_summary["order_id"].nunique())

Rows: 98673
Unique orders: 98673


In [23]:
final_df = final_df.merge(
    review_summary,
    on="order_id",
    how="left"
)

print("Final shape:", final_df.shape)

Final shape: (113425, 36)


In [24]:
final_df[[
    "order_id",
    "product_id",
    "price",
    "total_payment",
    "avg_review_score",
    "delivery_time_days"
]].head()

,order_id,product_id,price,total_payment,avg_review_score,delivery_time_days
0,e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,29.99,38.71,4.0,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,118.70,141.46,4.0,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,159.90,179.12,5.0,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,72.20,5.0,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,65266b2da20d04dbe00c5c2d3bb7859e,19.90,28.62,5.0,2.0


In [25]:
print("Final shape:", final_df.shape)

print("\nDuplicate rows:")
print(final_df.duplicated().sum())

print("\nMissing values:")
print(final_df.isnull().sum().sort_values(ascending=False).head(10))

Final shape: (113425, 36)

Duplicate rows:
0

Missing values:
delivery_delay_days              3229
delivery_time_days               3229
order_delivered_customer_date    3229
product_category_name            2378
product_name_lenght              2378
product_description_lenght       2378
product_photos_qty               2378
order_delivered_carrier_date     1968
review_count                      961
avg_review_score                  961
dtype: int64


In [26]:
print("Unique orders:", final_df["order_id"].nunique())
print("Unique customers:", final_df["customer_id"].nunique())
print("Unique products:", final_df["product_id"].nunique())

print("Total item value:", final_df["price"].sum())

Unique orders: 99441
Unique customers: 99441
Unique products: 32951
Total item value: 13591643.7


In [27]:
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

final_df.to_csv(
    PROCESSED_PATH / "olist_analysis.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!
